# Data Preprocessing

Flatten collected YouTube comments and replies into rows for sentiment analysis and network analysis.


In [ ]:
import json
from pathlib import Path
import pandas as pd
import random 
from langdetect import detect, LangDetectException
from collections import Counter
import nltk
import string
import re
import html
import emoji
from nltk.corpus import stopwords
from emoji

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
VIDEO_DATA_PATH = DATA_DIR / "video_data.json"
FLATTENED_COMMENTS_JSON = DATA_DIR / "comments_flattened.json"
FLATTENED_COMMENTS_CSV = DATA_DIR / "comments_flattened.csv"

RANDOM_SEED = 42


In [27]:
with open(VIDEO_DATA_PATH, "r", encoding="utf-8") as f:
    video_data = json.load(f)["videos"]

print("Video data loaded")
print("Total videos:", len(video_data))
print("Total collected comment rows:", sum(len(video.get("comments", [])) for video in video_data))


Video data loaded
Total videos: 110
Total collected comment rows: 63250


In [ ]:
comments_flattened = []
for video in video_data:
    video_context = {
        "video_id": video.get("videoId"),
        "video_title": video.get("title"),
        "channel_id": video.get("channelId"),
        "channel_title": video.get("channelTitle"),
        "video_published_at": video.get("publishedAt"),
        "video_view_count": video.get("viewCount", 0),
        "video_like_count": video.get("likeCount", 0),
        "video_available_comment_count": video.get("commentCount", 0),
    }

    for comment in video.get("comments", []):
        comments_flattened.append({
            **video_context,
            "comment_id": comment.get("commentId"),
            "comment_text": comment.get("text", ""),
            "comment_author_id": comment.get("authorId"),
            "comment_author": comment.get("author"),
            "comment_published_at": comment.get("publishedAt"),
            "comment_updated_at": comment.get("updatedAt"),
            "comment_like_count": comment.get("likeCount", 0),
            "is_reply": comment.get("isReply", False),
            "parent_comment_id": comment.get("parentCommentId"),
            "reply_to_author_id": comment.get("replyToAuthorId"),
            "top_level_reply_count": comment.get("totalReplyCount", 0),
            "text_length": len(comment.get("text", "") or ""),
        })

total_comments = len(comments_flattened)
total_replies = sum(1 for c in comments_flattened if c.get("is_reply") == True)
total_parent_comments = total_comments - total_replies

print(f"Flattened comment rows: {total_comments}\n")

print(f"Total comments: {total_comments}")
print(f"Total parent comments: {total_parent_comments}")
print(f"Total replies: {total_replies}")


Flattened comment rows: 63250

Total comments: 63250
Total parent comments: 48960
Total replies: 14290


In [ ]:

TOTAL_RANDOM_SAMPLES = 25

print("\nRANDOM SAMPLE OF COMMENTS TO IDENTIFY ISSUES")
print("=" * 80)
random.seed(RANDOM_SEED)
random_sample_indices = random.sample(range(len(comments_flattened)), min(TOTAL_RANDOM_SAMPLES, len(comments_flattened)))
for i, idx in enumerate(random_sample_indices):
    text = comments_flattened[idx]['comment_text'].strip().replace('\n', ' ').replace('\r', '')
    text = ' '.join(text.split())
    print(f"[{i+1}/{TOTAL_RANDOM_SAMPLES}] {text[:150]:<10}")



RANDOM SAMPLE OF COMMENTS TO IDENTIFY ISSUES
[1/25] Can’t be bothered to pronounce the Asian names properly I guess.
[2/25] 14:30 in my opinion, it needed a big sumptuous cloak, maybe with a hood and gloves, to be more evocative of how luscious and involved klimt’s pieces a
[3/25] what is cara doing :/
[4/25] We need you at the Grammy asap😂😂
[5/25] @LadyAxe13 🤣
[6/25] Megyn is sooo jealous she wasn't invited
[7/25] My only look at the met gala, thanks Garrron! Getting Halloween in springtime vibes 🤡👻👽👀
[8/25] I agree with so many of your critiques. The fact Anna didn't even bother with the theme and wore a variation of a previous dress let's me know how fri
[9/25] I be sick of them shades. You can't wear shades with everything. I was just thinking this when I first seen this video. She look like one of the actor
[10/25] NINGNING  
[11/25] Vogue deletes comments and leaves hates comments on him.
[12/25] Exactly!  
[13/25] The idea is worth sharing This deserves recognition.
[14/25] Bla

In [ ]:
def detect_language(text):
    """Detect language, returning ISO code."""
    try:
        if not text or not len(text.strip()):
            return 'en'
        return detect(text)
    except LangDetectException:
        return 'unknown'

# Collect comment rows from the rows list
language_results = [(comment, detect_language(comment.get('comment_text', ''))) for comment in comments_flattened]
language_counter = Counter(lang for _, lang in language_results)

In [54]:
TOP_LANGUAGE_COUNT = 10
TOTAL_NON_ENGLISH_EXAMPLES = 20

total_comments = len(language_results)

print(f"\nTOP {TOP_LANGUAGE_COUNT} LANGUAGE DETECTIONS")
print("=" * 80)
for i, (lang, count) in enumerate(language_counter.most_common(TOP_LANGUAGE_COUNT), start=1):
    pct = 100 * count / total_comments
    print(f"[{i}] {lang} {count:,} ({pct:.2f}%)")
    if i == 10:
        break


TOP 10 LANGUAGE DETECTIONS
[1] en 46,946 (74.22%)
[2] unknown 1,679 (2.65%)
[3] so 1,543 (2.44%)
[4] pt 1,152 (1.82%)
[5] tl 1,061 (1.68%)
[6] de 1,061 (1.68%)
[7] af 979 (1.55%)
[8] fr 842 (1.33%)
[9] et 814 (1.29%)
[10] id 781 (1.23%)


In [51]:
COMMENT_TRUNCATION_LENGTH = 200

# Extract full comment data for English comments
english_comments = [comment for comment, lang in language_results if lang == 'en']

# Extract truncated comment data for non-English comments for test display
non_english_comments = [comment['comment_text'][:COMMENT_TRUNCATION_LENGTH] for comment, lang in language_results if lang != 'en']

random.seed(RANDOM_SEED)
random_samples = random.sample(non_english_comments, min(TOTAL_NON_ENGLISH_EXAMPLES, len(non_english_comments)))

print("\nRANDOM NON-ENGLISH COMMENTS REMOVED:")
print("=" * 80)
for idx, text in enumerate(random_samples):
    print(f"[{idx+1}/{TOTAL_NON_ENGLISH_EXAMPLES}] {text}")



RANDOM NON-ENGLISH COMMENTS REMOVED:
[1/20] Itne bade bhi nahi hai tumhare ki dono hath lagane pade nimbu hai kacche wale😂😂😂😂
[2/20] Jisooo ❤❤
[3/20] I love Lisa so much
[4/20] Umili e gentile questo ragazzo  merita fare successo ❤
[5/20] 🤦🏼‍♀️
[6/20] 🤣🤣🤣❤️
[7/20] cosplay event
[8/20] ITS BLACKPINK GALA
[9/20] Нет.. Это не МАЙКЛ((!  НЕНАДО ДРУГОГО НАМ
[10/20] Rosé ❤❤❤
[11/20] Espectacular el vestido !!!!
[12/20] Very handsome
[13/20] And???
[14/20] ok didn't see Janelle Monae yet, it's a tie. 😮
[15/20] ❤❤❤
[16/20] ✨ L
[17/20] Agenda detected
[18/20] F Amazon
[19/20] في عرببببب😭
[20/20] @Mrkarki0004 CANNABLES


In [ ]:
english_count = len(english_comments)
removed_count = total_comments - english_count
english_pct = 100 * english_count / total_comments
removed_pct = 100 * removed_count / total_comments

print(f"\nEnglish kept: {english_count} ({english_pct:.1f}%)")
print(f"Non-English removed: {removed_count} ({removed_pct:.1f}%)")


English kept: 46946 (74.2%)
Non-English removed: 16304 (25.8%)


In [ ]:
# Regex patterns for text cleaning
URL_PATTERN = re.compile(r'https?://\S+') # Remove URLs
TIMESTAMP_PATTERN = re.compile(r'\b\d{1,2}:\d{2}(?::\d{2})?\b') # Remove timestamps
MENTION_PATTERN = re.compile(r'@[\w.-]+[\w]') # Remove mentions
DIGIT_PATTERN = re.compile(r'\d+') # Remove digits
PUNCT_PATTERN = re.compile(r'[^\w\s]') # Remove any punctuation 

In [ ]:
unique_videos = set(comment['video_id'] for comment in comments_flattened)
unique_channels = set(comment['channel_id'] for comment in comments_flattened)
unique_authors = set(comment['comment_author'] for comment in comments_flattened)

SHORT_COMMENT_LENGTH = 10
short_comments = [comment for comment in comments_flattened if len(comment['comment_text']) < SHORT_COMMENT_LENGTH]
comment_lengths = [len(comment['comment_text']) for comment in comments_flattened]

comments_with_urls = [c for c in comments_flattened if URL_PATTERN.search(c['comment_text'])]
comments_with_timestamps = [c for c in comments_flattened if TIMESTAMP_PATTERN.search(c['comment_text'])]
comments_with_mentions = [c for c in comments_flattened if MENTION_PATTERN.search(c['comment_text'])]
comments_with_digits = [c for c in comments_flattened if DIGIT_PATTERN.search(c['comment_text'])]
comments_with_punctuation = [c for c in comments_flattened if PUNCT_PATTERN.search(c['comment_text'])]

comments_with_urls_pct = 100 * len(comments_with_urls) / len(comments_flattened)
comments_with_timestamps_pct = 100 * len(comments_with_timestamps) / len(comments_flattened)
comments_with_mentions_pct = 100 * len(comments_with_mentions) / len(comments_flattened)
comments_with_digits_pct = 100 * len(comments_with_digits) / len(comments_flattened)
comments_with_punctuation_pct = 100 * len(comments_with_punctuation) / len(comments_flattened)

# For author, video, and channel distributions
video_counter = Counter(comment['video_id'] for comment in comments_flattened)
channel_counter = Counter(comment['channel_id'] for comment in comments_flattened)
author_counter = Counter(comment['comment_author'] for comment in comments_flattened)

# Print all details at bottom
print("\nDATASET DETAILS")
print("=" * 80)
print(f"Total comments: {len(comments_flattened)}")
print(f"Unique videos: {len(unique_videos)}")
print(f"Unique channels: {len(unique_channels)}")
print(f"Unique authors: {len(unique_authors)}\n")

print(f"Short comments (<{SHORT_COMMENT_LENGTH} chars): {len(short_comments)}")
print(f"Comments w/ URLs: {len(comments_with_urls)} ({comments_with_urls_pct:.2f}%)")
print(f"Comments w/ timestamps: {len(comments_with_timestamps)} ({comments_with_timestamps_pct:.2f}%)")
print(f"Comments w/ mentions: {len(comments_with_mentions)} ({comments_with_mentions_pct:.2f}%)")
print(f"Comments w/ digits: {len(comments_with_digits)} ({comments_with_digits_pct:.2f}%)")
print(f"Comments w/ punctuation: {len(comments_with_punctuation)}\n")

print(f"Min comment length: {min(comment_lengths) if comment_lengths else 0}")
print(f"Max comment length: {max(comment_lengths) if comment_lengths else 0}")
print(f"Average comment length: {sum(comment_lengths)/len(comment_lengths):.2f}" if comment_lengths else "Avg. comment length: 0")



DATASET DETAILS
Total comments: 63250
Unique videos: 110
Unique channels: 74
Unique authors: 46504

Short comments (<10 chars): 4626
Comments w/ URLs: 26 (0.04%)
Comments w/ timestamps: 2368 (3.74%)
Comments w/ mentions: 4592 (7.26%)
Comments w/ digits: 8905 (14.08%)
Comments w/ punctuation: 51910

Min comment length: 0
Max comment length: 9817
Average comment length: 80.41


In [ ]:
TWEET_TOKENISER = nltk.tokenize.TweetTokenizer(
    reduce_len=True,
    strip_handles=True,
    preserve_case=False 
)

PUNCTUATION = list(string.punctuation)
TWEET_STEMMER = nltk.stem.PorterStemmer()
STOP_WORDS = set(stopwords.words('english')) | set(PUNCTUATION)


In [72]:

def remove_html_entities(text):
    return html.unescape(text)

def remove_urls(text):
    return URL_PATTERN.sub('', text)

def remove_timestamps(text):
    return TIMESTAMP_PATTERN.sub('', text)

def remove_mentions(text):
    return MENTION_PATTERN.sub('', text)

def remove_digits(text):
    return DIGIT_PATTERN.sub(' ', text)

def normalise_whitespace(text):
    return ' '.join(text.split())

def remove_unicode(text):
    return text.encode("ascii", "ignore").decode()

def strip_punctuation(text):
    return PUNCT_PATTERN.sub(' ', text)

def tokenize(text):
    return TWEET_TOKENISER.tokenize(text)

def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOP_WORDS]

def stem_tokens(tokens):
    return [TWEET_STEMMER.stem(t) for t in tokens]

def remove_emojis(text):
    return emoji.replace_emoji(text, replace='')

def purify_text(text, show_changes=False):
    """
    Heavy preprocessing pipeline suitable for lexical-based sentiment analysis
    If show_changes is True, returns list of text at different stages
    """
    def pipeline(t):
        t = t.lower().strip()
        t = remove_html_entities(t)
        t = remove_unicode(t)
        t = remove_timestamps(t)
        t = remove_urls(t)
        t = remove_mentions(t)
        t = remove_digits(t)
        t = strip_punctuation(t)
        t = normalise_whitespace(t)
        tokens = tokenize(t)
        tokens = remove_stopwords(tokens)
        tokens = stem_tokens(tokens)
        return tokens

    if show_changes:
        history = {}
        t = text
        t = t.lower().strip()
        history["lowercase_and_strip"] = t
        t = remove_html_entities(t)
        history["remove_html_entities"] = t
        t = remove_unicode(t)
        history["remove_unicode"] = t
        t = remove_timestamps(t)
        history["remove_timestamps"] = t
        t = remove_urls(t)
        history["remove_urls"] = t
        t = remove_mentions(t)
        history["remove_mentions"] = t
        t = remove_digits(t)
        history["remove_digits"] = t
        t = strip_punctuation(t)
        history["strip_punctuation"] = t
        t = normalise_whitespace(t)
        history["normalise_whitespace"] = t
        t = tokenize(t)
        history["tokenize"] = t
        t = remove_stopwords(t)
        history["remove_stopwords"] = t
        t = stem_tokens(t)
        history["stem_tokens"] = t
        return history
    else:
        return pipeline(text)
   
   
    


In [ ]:
RANDOM_TOTAL_EXAMPLES = 10

purify_random_samples = random.sample(english_comments, RANDOM_TOTAL_EXAMPLES)
for idx, text in enumerate(purify_random_samples):
    history = purify_text(text['comment_text'], True).items()
    print(f"\n[{idx+1}/{RANDOM_TOTAL_EXAMPLES}] {text['comment_text'][:100]}")
    for step, value in history:
        print(f"  [{step}] {value}")


[1/10] Take me to your leader
  [lowercase_and_strip] take me to your leader
  [remove_html_entities] take me to your leader
  [remove_unicode] take me to your leader
  [remove_timestamps] take me to your leader
  [remove_urls] take me to your leader
  [remove_mentions] take me to your leader
  [remove_digits] take me to your leader
  [strip_punctuation] take me to your leader
  [normalise_whitespace] take me to your leader
  [tokenize] ['take', 'me', 'to', 'your', 'leader']
  [remove_stopwords] ['take', 'leader']
  [stem_tokens] ['take', 'leader']

[2/10] Cher covered up her body because she's 80 years old. There's major creppy wrinkly skin going on. She
  [lowercase_and_strip] cher covered up her body because she's 80 years old. there's major creppy wrinkly skin going on. she doesn't want that kind of talk.
  [remove_html_entities] cher covered up her body because she's 80 years old. there's major creppy wrinkly skin going on. she doesn't want that kind of talk.
  [remove_unicode] c